# Линейные модели и методы оптимизации — Интерактивный конспект

**Темы:**
- Градиентный спуск (gradient descent)
- Логистическая регрессия (logistic regression)
- Регуляризация: L1 (Lasso), L2 (Ridge), Elastic Net
- Стохастический градиентный спуск (SGD) с мини-батчами

Конспект самодостаточный — все примеры можно запускать последовательно.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

%matplotlib inline
np.random.seed(42)

---

## 1. Градиентный спуск

### Идея

Градиентный спуск — итеративный алгоритм поиска минимума функции. На каждом шаге мы двигаемся в направлении **антиградиента** (наибольшего убывания функции):

$$w_{t+1} = w_t - \eta \cdot \nabla f(w_t)$$

где $\eta$ — **learning rate** (скорость обучения).

### Градиент

Градиент — вектор частных производных функции по каждой переменной. Он указывает направление наибольшего роста функции.

Для функции $f(x, y) = \sin^2(x) + \sin^2(y)$:

$$\nabla f = \left(\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}\right) = (\sin 2x, \sin 2y)$$

Вспомним, что $\frac{d}{dx}\sin^2(x) = 2\sin(x)\cos(x) = \sin(2x)$ (по цепному правилу).

In [ ]:
# Определяем функцию и её градиент
def f(r):
    """f(x, y) = sin²(x) + sin²(y)"""
    return np.sum(np.sin(r) ** 2)

def grad_f(r):
    """Градиент: (sin(2x), sin(2y))"""
    return np.array([np.sin(2 * r[0]), np.sin(2 * r[1])])

# Проверка: в точке (1, 2) градиент должен быть [sin(2), sin(4)]
r_test = np.array([1.0, 2.0])
print(f"f({r_test}) = {f(r_test):.4f}")
print(f"grad_f({r_test}) = {grad_f(r_test)}")
print(f"Ожидаемое: [sin(2), sin(4)] = [{np.sin(2):.4f}, {np.sin(4):.4f}]")

### Реализация градиентного спуска 2D

Алгоритм:
1. Стартуем из случайной точки $r_0$
2. На каждой итерации: $r \leftarrow r - \eta \cdot \nabla f(r)$
3. Записываем историю $(x, y, f(x,y))$

In [ ]:
def grad_descent_2d(f, grad_f, lr, num_iter=100, r0=None):
    """
    Градиентный спуск для функции от двух переменных.
    Возвращает историю: массив [num_iter, 3] — (x, y, f(x,y)).
    """
    if r0 is None:
        r0 = np.random.random(2)

    history = []
    curr_r = r0.copy()

    for _ in range(num_iter):
        history.append(np.hstack((curr_r, f(curr_r))))
        curr_r -= lr * grad_f(curr_r)  # ← ключевая строка!

    return np.vstack(history)

# Запускаем из точки (1.0, 1.5) с lr=0.1
steps = grad_descent_2d(f, grad_f, lr=0.1, num_iter=30, r0=np.array([1.0, 1.5]))
print(f"Начало: f({steps[0, :2]}) = {steps[0, 2]:.4f}")
print(f"Конец:  f({steps[-1, :2]}) = {steps[-1, 2]:.6f}")

In [ ]:
# Визуализация: как функция убывает на каждом шаге
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: значение функции от номера итерации
axes[0].plot(steps[:, 2], 'b-o', markersize=4)
axes[0].set_xlabel('Номер итерации')
axes[0].set_ylabel('$f(x, y)$')
axes[0].set_title('Убывание функции при градиентном спуске')
axes[0].grid(True)

# График 2: траектория на контурной карте
x_range = np.linspace(-1.5, 1.5, 200)
y_range = np.linspace(-1.5, 1.5, 200)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = np.sin(X_grid)**2 + np.sin(Y_grid)**2

axes[1].contourf(X_grid, Y_grid, Z_grid, levels=30, cmap='viridis', alpha=0.8)
axes[1].plot(steps[:, 0], steps[:, 1], 'r-o', markersize=5, label='Траектория')
axes[1].plot(steps[0, 0], steps[0, 1], 'r*', markersize=15, label='Старт')
axes[1].plot(steps[-1, 0], steps[-1, 1], 'gs', markersize=10, label='Финиш')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Траектория градиентного спуска')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Влияние learning rate

Learning rate — ключевой гиперпараметр:
- **Слишком маленький** $\eta$ — медленная сходимость
- **Слишком большой** $\eta$ — расходимость (значения «прыгают»)
- **Оптимальный** $\eta$ — быстрая и устойчивая сходимость

In [ ]:
# Сравнение разных learning rate
r0 = np.array([1.0, 1.5])
learning_rates = [0.01, 0.1, 0.5, 0.95]

plt.figure(figsize=(12, 5))
for lr in learning_rates:
    s = grad_descent_2d(f, grad_f, lr=lr, num_iter=30, r0=r0)
    plt.plot(s[:, 2], '-o', markersize=3, label=f'lr={lr}')

plt.xlabel('Номер итерации')
plt.ylabel('$f(x, y)$')
plt.title('Влияние learning rate на сходимость')
plt.legend()
plt.grid(True)
plt.show()

---

## 2. Сигмоидная функция и логистическая регрессия

### Сигмоида

Сигмоидная функция преобразует любое вещественное число в вероятность (интервал $(0, 1)$):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Свойства:**
- $\sigma(0) = 0.5$
- $\sigma(z) \to 1$ при $z \to +\infty$, $\sigma(z) \to 0$ при $z \to -\infty$
- Производная: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$

In [ ]:
# Вспомогательные функции
def sigmoid(h):
    """Сигмоидная функция"""
    return 1. / (1 + np.exp(-h))

def logit(x, w):
    """Линейная комбинация: z = X @ w"""
    return np.dot(x, w)

# Визуализация сигмоиды
z = np.linspace(-8, 8, 200)
plt.figure(figsize=(10, 5))
plt.plot(z, sigmoid(z), 'b-', linewidth=2, label=r'$\sigma(z)$')
plt.plot(z, sigmoid(z) * (1 - sigmoid(z)), 'r--', linewidth=2, label=r"$\sigma'(z) = \sigma(z)(1 - \sigma(z))$")
plt.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('z')
plt.ylabel('$\sigma(z)$')
plt.title('Сигмоидная функция и её производная')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

### Логистическая регрессия — принцип работы

Логистическая регрессия предсказывает вероятность принадлежности к классу 1:

$$p(y=1|x) = \sigma(w^T x) = \frac{1}{1 + e^{-w^T x}}$$

**Функция потерь** — кросс-энтропия (log-loss):

$$L(w) = -\sum_{i=1}^{N} \left[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \right]$$

Интуиция: штрафуем модель за уверенные неправильные предсказания. Если $y_i = 1$, а $p_i \approx 0$ — штраф огромный ($-\log(0) \to \infty$).

**Градиент** имеет удивительно простую форму:

$$\nabla_w L = X^T (p - y)$$

где $p$ — вектор предсказанных вероятностей, $y$ — вектор истинных меток.

---

## 3. Мини-батчевый SGD: генератор батчей

При стохастическом градиентном спуске мы вычисляем градиент не по всей выборке, а по случайным подмножествам (батчам). Это:
- **Быстрее** — не нужно обрабатывать все данные для одного шага
- **Добавляет шум** — помогает выходить из плохих локальных минимумов
- **Экономит память** — не нужно хранить все данные одновременно

Важный нюанс: **последний неполный батч отбрасывается** — это стандартная практика для стабильности обучения.

In [ ]:
def generate_batches(X, y, batch_size):
    """
    Генератор мини-батчей для SGD.
    Перемешивает данные и выдаёт батчи фиксированного размера.
    Последний неполный батч отбрасывается.
    """
    assert len(X) == len(y)
    np.random.seed(42)
    X = np.array(X)
    y = np.array(y)
    perm = np.random.permutation(len(X))

    for batch_start in range(0, len(X) - batch_size + 1, batch_size):
        batch_indices = perm[batch_start:batch_start + batch_size]
        yield X[batch_indices], y[batch_indices]

# Демонстрация: 100 объектов, батч = 30 → 3 полных батча (10 объектов отбрасываются)
X_demo = np.arange(100).reshape(-1, 1)
y_demo = np.arange(100)

num_batches = 0
for X_b, y_b in generate_batches(X_demo, y_demo, 30):
    num_batches += 1
    print(f"Батч {num_batches}: размер = {len(X_b)}, первые 5 индексов = {y_b[:5]}")

print(f"\nВсего батчей: {num_batches} (100 // 30 = {100 // 30}, остаток {100 % 30} отброшен)")

---

## 4. Реализация логистической регрессии с нуля

### Архитектура класса

Ключевые моменты реализации:
- **Bias (сдвиг)** добавляется через колонку единиц слева: `X_train = [1 | X]`
- Веса инициализируются случайно: `np.random.randn(n + 1)` (+1 для bias)
- Градиент — **сумма** по батчу, не среднее (это важно для совместимости с тестами)
- Лосс записывается на **каждом батче**, не усредняется по эпохе
- Наследование от `BaseEstimator, ClassifierMixin` даёт совместимость со sklearn

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin


class MyLogisticRegression(BaseEstimator, ClassifierMixin):
    """Логистическая регрессия, реализованная с нуля."""

    def __init__(self):
        self.w = None

    def fit(self, X, y, epochs=10, lr=0.1, batch_size=100):
        """
        Обучение методом мини-батчевого SGD.

        Шаги:
        1. Добавляем колонку единиц (для bias)
        2. Инициализируем веса случайно
        3. Для каждой эпохи, для каждого батча:
           a. Предсказываем вероятности: p = sigmoid(X @ w)
           b. Считаем лосс: L = -sum(y*log(p) + (1-y)*log(1-p))
           c. Считаем градиент: grad = X^T @ (p - y)
           d. Обновляем веса: w -= lr * grad
        """
        l, n = X.shape
        if self.w is None:
            np.random.seed(42)
            self.w = np.random.randn(n + 1)

        # Добавляем колонку единиц для bias
        X_train = np.concatenate((np.ones((l, 1)), X), axis=1)

        losses = []
        for epoch in range(epochs):
            for X_batch, y_batch in generate_batches(X_train, y, batch_size):
                # Прямой проход
                predictions = sigmoid(logit(X_batch, self.w))
                # Считаем и записываем лосс
                loss = self.__loss(y_batch, predictions)
                losses.append(loss)
                # Обновляем веса
                self.w -= lr * self.get_grad(X_batch, y_batch, predictions)

        self.is_fitted_ = True
        return losses

    def get_grad(self, X_batch, y_batch, predictions):
        """
        Градиент кросс-энтропии: X^T @ (p - y)
        Это СУММА, не среднее!
        """
        return X_batch.T @ (predictions - y_batch)

    def predict_proba(self, X):
        """Предсказание вероятностей (добавляем колонку единиц)."""
        l, n = X.shape
        X_ = np.concatenate((np.ones((l, 1)), X), axis=1)
        return sigmoid(logit(X_, self.w))

    def predict(self, X, threshold=0.5):
        """Предсказание классов по порогу 0.5."""
        return (self.predict_proba(X) >= threshold).astype(int)

    def get_weights(self):
        return self.w.copy()

    def __loss(self, y, p):
        """Кросс-энтропия (log-loss). clip для численной стабильности."""
        p = np.clip(p, 1e-10, 1 - 1e-10)
        return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

print("Класс MyLogisticRegression определён.")

### Тестируем на искусственных данных

Создадим два кластера и посмотрим, как модель находит разделяющую границу.

In [ ]:
from sklearn.datasets import make_blobs

# Создаём два линейно разделимых кластера
X_blobs, y_blobs = make_blobs(n_samples=500, centers=[[-2, 0.5], [3, -0.5]],
                               cluster_std=1, random_state=42)

# Обучаем нашу модель
clf = MyLogisticRegression()
losses = clf.fit(X_blobs, y_blobs, epochs=500, lr=0.01, batch_size=50)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: убывание лосса
axes[0].plot(losses)
axes[0].set_xlabel('Шаг (батч)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Убывание функции потерь при обучении')
axes[0].grid(True)

# График 2: разделяющая поверхность
eps = 0.5
xx, yy = np.meshgrid(
    np.linspace(X_blobs[:, 0].min() - eps, X_blobs[:, 0].max() + eps, 200),
    np.linspace(X_blobs[:, 1].min() - eps, X_blobs[:, 1].max() + eps, 200)
)
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
axes[1].pcolormesh(xx, yy, Z, cmap=cmap_light)
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1],
                c=['red' if yi == 0 else 'green' for yi in y_blobs], s=15)
axes[1].set_title('Разделяющая поверхность')
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

print(f"Веса модели: {clf.get_weights()}")

---

## 5. Регуляризация: L1, L2 и Elastic Net

### Зачем нужна регуляризация?

Без регуляризации модель может **переобучиться** — выучить шум в данных, а не закономерности. Регуляризация штрафует большие веса, заставляя модель быть проще.

### L1 (Lasso) — штраф за абсолютные значения весов

$$L_{L1} = \alpha \sum_{j=1}^{n} |w_j|$$

- Градиент: $\nabla L_{L1} = \alpha \cdot \text{sign}(w)$
- Эффект: обнуляет веса незначимых признаков → **отбор признаков**
- $\text{sign}(w)$ = $+1$ если $w > 0$, $-1$ если $w < 0$, $0$ если $w = 0$

### L2 (Ridge) — штраф за квадраты весов

$$L_{L2} = \beta \sum_{j=1}^{n} w_j^2$$

- Градиент: $\nabla L_{L2} = 2\beta \cdot w$
- Эффект: уменьшает все веса пропорционально → **стабильность**
- Не обнуляет веса полностью, но делает их маленькими

### Elastic Net — комбинация L1 + L2

$$L_{total} = L_{base} + \alpha \sum |w_j| + \beta \sum w_j^2$$

Комбинирует преимущества обоих подходов.

### Важно: bias не регуляризуется!

$w_0$ (bias, сдвиг) не входит в регуляризационные слагаемые. Регуляризация bias сместила бы предсказания и ухудшила модель.

In [ ]:
class MyElasticLogisticRegression(MyLogisticRegression):
    """Логистическая регрессия с Elastic Net регуляризацией."""

    def __init__(self, l1_coef=0.01, l2_coef=0.01):
        self.l1_coef = l1_coef
        self.l2_coef = l2_coef
        self.w = None

    def _MyLogisticRegression__loss(self, y, p):
        """
        Переопределяем приватный __loss родителя.
        Python использует name mangling: __loss → _ClassName__loss
        """
        p = np.clip(p, 1e-10, 1 - 1e-10)
        base_loss = -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        # Регуляризация: w[1:] — исключаем bias (w[0])
        l1_penalty = self.l1_coef * np.sum(np.abs(self.w[1:]))
        l2_penalty = self.l2_coef * np.sum(self.w[1:] ** 2)
        return base_loss + l1_penalty + l2_penalty

    def get_grad(self, X_batch, y_batch, predictions):
        """
        Градиент с L1 + L2 регуляризацией.
        Bias (w[0]) не регуляризуется.
        """
        # Базовый градиент
        grad_basic = X_batch.T @ (predictions - y_batch)

        # L1: alpha * sign(w), с обнулённым bias
        grad_l1 = self.l1_coef * np.sign(self.w)
        grad_l1[0] = 0  # bias не регуляризуется

        # L2: 2 * beta * w, с обнулённым bias
        grad_l2 = 2 * self.l2_coef * self.w
        grad_l2[0] = 0  # bias не регуляризуется

        return grad_basic + grad_l1 + grad_l2

print("Класс MyElasticLogisticRegression определён.")

### Визуализация эффекта регуляризации

Сравним веса модели без регуляризации и с разными коэффициентами:

In [ ]:
# Сравнение: без регуляризации vs с регуляризацией
configs = [
    ("Без регуляризации", MyLogisticRegression()),
    ("Elastic Net (0.01, 0.01)", MyElasticLogisticRegression(0.01, 0.01)),
    ("Elastic Net (0.1, 0.1)", MyElasticLogisticRegression(0.1, 0.1)),
    ("Elastic Net (1.0, 1.0)", MyElasticLogisticRegression(1.0, 1.0)),
]

fig, axes = plt.subplots(1, len(configs), figsize=(18, 4))
for ax, (name, model) in zip(axes, configs):
    model.fit(X_blobs, y_blobs, epochs=200, lr=0.01, batch_size=50)
    w = model.get_weights()

    # Разделяющая поверхность
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA'])
    ax.pcolormesh(xx, yy, Z, cmap=cmap_light)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1],
               c=['red' if yi == 0 else 'green' for yi in y_blobs], s=8)
    ax.set_title(f"{name}\nw={np.round(w, 2)}")

plt.tight_layout()
plt.show()

---

## 6. Применение на реальных данных: MNIST (0 vs 1)

### Стандартизация и Pipeline

Для линейных моделей **стандартизация признаков критически важна**. `StandardScaler` приводит каждый признак к нулевому среднему и единичной дисперсии:

$$x' = \frac{x - \mu}{\sigma}$$

`make_pipeline` из sklearn позволяет объединить предобработку и модель в единый объект, который:
- Автоматически вызывает `fit_transform` на обучающих данных
- Автоматически вызывает `transform` на тестовых данных
- Предотвращает data leakage при кросс-валидации

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)  # подавляем overflow в exp

# Загрузка данных MNIST
data = pd.read_csv('data/train.csv')

# Выбираем только цифры 0 и 1
X_mnist = data.iloc[:, 1:]
y_mnist = data.iloc[:, 0]
mask = (y_mnist == 0) | (y_mnist == 1)
X_mnist = X_mnist[mask].values
y_mnist = y_mnist[mask].values

print(f"Размер выборки: {X_mnist.shape}")
print(f"Классы: 0 — {(y_mnist == 0).sum()} шт., 1 — {(y_mnist == 1).sum()} шт.")

In [ ]:
# Посмотрим на несколько примеров из датасета
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_mnist[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"label={y_mnist[i]}")
    ax.axis('off')
plt.suptitle('Примеры изображений из MNIST (28x28 = 784 пикселя)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Pipeline: StandardScaler → MyElasticLogisticRegression
pipe = make_pipeline(
    StandardScaler(),
    MyElasticLogisticRegression(l1_coef=0.01, l2_coef=0.01)
)

# Кросс-валидация (5 фолдов)
scores = cross_val_score(pipe, X_mnist, y_mnist, scoring='accuracy', cv=5)

print(f"Accuracy по фолдам: {scores}")
print(f"Средняя accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

---

## 7. Попробуй сам!

### Задание 1: Эксперимент с learning rate

Попробуй запустить градиентный спуск для функции $f(x,y) = \sin^2(x) + \sin^2(y)$ с разными `lr` и `num_iter`. Что произойдёт, если `lr = 1.0`? А если `lr = 0.001`?

> **Подсказка**: используй функцию `grad_descent_2d` и постройте графики.

In [ ]:
# Твой код здесь:
# steps_large_lr = grad_descent_2d(f, grad_f, lr=???, num_iter=50, r0=np.array([1.0, 1.5]))
# plt.plot(steps_large_lr[:, 2])
# plt.title("lr = ???")
# plt.show()

### Задание 2: Влияние коэффициентов регуляризации

Попробуй обучить `MyElasticLogisticRegression` на данных `make_blobs` с разными значениями `l1_coef` и `l2_coef`. Как меняются веса модели?

> **Подсказка**: большие значения L1 приводят к обнулению весов. Сравни `get_weights()` для разных конфигураций.

In [ ]:
# Твой код здесь:
# for l1, l2 in [(0, 0.01), (0.01, 0), (0.5, 0.5), (5.0, 5.0)]:
#     model = MyElasticLogisticRegression(l1_coef=l1, l2_coef=l2)
#     model.fit(X_blobs, y_blobs, epochs=200, lr=0.01, batch_size=50)
#     print(f"l1={l1}, l2={l2} → weights = {np.round(model.get_weights(), 3)}")

### Задание 3: Сравнение с sklearn

Сравни результаты нашей реализации с `LogisticRegression` из sklearn на MNIST (0 vs 1). Совпадает ли accuracy?

> **Подсказка**: `from sklearn.linear_model import LogisticRegression`, используй `cross_val_score` с тем же `cv=5`.

In [ ]:
# Твой код здесь:
# from sklearn.linear_model import LogisticRegression
# pipe_sklearn = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))
# scores_sklearn = cross_val_score(pipe_sklearn, X_mnist, y_mnist, scoring='accuracy', cv=5)
# print(f"sklearn accuracy: {scores_sklearn.mean():.4f}")

---

## Шпаргалка: ключевые формулы

| Компонент | Формула |
|-----------|---------|
| Сигмоида | $\sigma(z) = \frac{1}{1 + e^{-z}}$ |
| Кросс-энтропия | $L = -\sum [y \log p + (1-y) \log(1-p)]$ |
| Градиент логрегрессии | $\nabla L = X^T(p - y)$ |
| L1 штраф | $\alpha \sum \|w_j\|$, градиент: $\alpha \cdot \text{sign}(w)$ |
| L2 штраф | $\beta \sum w_j^2$, градиент: $2\beta \cdot w$ |
| Обновление весов | $w \leftarrow w - \eta \cdot \nabla L$ |
| Стандартизация | $x' = \frac{x - \mu}{\sigma}$ |

## Ссылки

- [Логистическая регрессия — sklearn docs](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)
- [StandardScaler — sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [make_pipeline — sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [Elastic Net — Wikipedia](https://en.wikipedia.org/wiki/Elastic_net_regularization)